# Exercise 2.3: Schema and Partition Evolution

In this exercise, you'll learn how Apache Iceberg handles table evolution:
- **Schema Evolution**: Add, remove, and rename columns without rewriting data
- **Field IDs**: Understand how Iceberg prevents data resurrection
- **Default Values**: Handle mismatches between old and new schemas
- **Partition Evolution**: Change partitioning strategies over time

## Learning Objectives
- Add and drop columns using metadata-only operations
- See how field IDs prevent accidentally resurrecting deleted data
- Use initial and write defaults for new columns
- Change partition specifications without rewriting data
- Understand the impact of partition evolution on query performance

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("SchemaAndPartitionEvolution") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.evolution")
print("Namespace 'evolution' created!")

## Part 1: Schema Evolution - Adding Columns

### Create Initial Table

In [ ]:
# Create a simple sales table
spark.sql("""
CREATE OR REPLACE TABLE polaris.evolution.sales (
    sale_id INT,
    product_name STRING,
    quantity INT,
    price DOUBLE,
    sale_date DATE
)
USING iceberg
""")

print("Sales table created!")

In [ ]:
# Insert initial data
spark.sql("""
INSERT INTO polaris.evolution.sales VALUES
    (1, 'Laptop', 2, 1200.00, DATE '2023-01-15'),
    (2, 'Mouse', 10, 25.00, DATE '2023-01-20'),
    (3, 'Keyboard', 5, 75.00, DATE '2023-02-10'),
    (4, 'Monitor', 3, 350.00, DATE '2023-02-15')
""")

print("Initial data inserted!")

In [ ]:
# View the schema
print("Initial schema:")
spark.sql("DESCRIBE polaris.evolution.sales").show()

### Add a Column - Instant Operation

In [ ]:
import time

# Time the operation
start = time.time()

# Add a customer_id column
spark.sql("""
ALTER TABLE polaris.evolution.sales
ADD COLUMN customer_id STRING
""")

elapsed = time.time() - start
print(f"Column added in {elapsed:.3f} seconds (metadata-only operation!)")

In [ ]:
# View updated schema
print("Schema after adding column:")
spark.sql("DESCRIBE polaris.evolution.sales").show()

In [ ]:
# Query the table - old rows have NULL for new column
print("Data after adding column:")
spark.sql("SELECT * FROM polaris.evolution.sales").show()

### Add Column with Default Value

In [ ]:
# Add discount_applied column (without default for now)
spark.sql("""
ALTER TABLE polaris.evolution.sales
ADD COLUMN discount_applied BOOLEAN
""")

print("Column added!")

In [ ]:
# Old rows have NULL for the new column (no defaults in Spark)
print("Old rows have NULL for new column:")
spark.sql("""
SELECT sale_id, product_name, discount_applied 
FROM polaris.evolution.sales
""").show()

### Insert New Data

In [ ]:
# Insert new rows with all columns
spark.sql("""
INSERT INTO polaris.evolution.sales VALUES
    (5, 'Headphones', 7, 89.99, DATE '2023-03-01', 'CUST001', true),
    (6, 'Webcam', 4, 120.00, DATE '2023-03-05', 'CUST002', false)
""")

print("New data inserted!")

In [ ]:
# View all data
print("All data:")
spark.sql("SELECT * FROM polaris.evolution.sales ORDER BY sale_id").show()

## Part 2: Field IDs and Data Resurrection Prevention

### View Field IDs

Iceberg uses internal field IDs to track columns, not names.

In [ ]:
# View the table's schema with field IDs
print("Schema with field IDs:")
spark.sql("DESCRIBE EXTENDED polaris.evolution.sales").show(truncate=False)

### Drop a Column

In [ ]:
# Drop the customer_id column
spark.sql("""
ALTER TABLE polaris.evolution.sales
DROP COLUMN customer_id
""")

print("customer_id column dropped!")

In [ ]:
# Verify it's gone
print("Schema after dropping column:")
spark.sql("DESCRIBE polaris.evolution.sales").show()

In [ ]:
# Data no longer shows customer_id
print("Data after dropping column:")
spark.sql("SELECT * FROM polaris.evolution.sales WHERE sale_id IN (5, 6)").show()

### Re-add Column with Same Name

This is where field IDs shine - the old data won't be resurrected!

In [ ]:
# Add customer_id back
spark.sql("""
ALTER TABLE polaris.evolution.sales
ADD COLUMN customer_id STRING
""")

print("customer_id column re-added!")

In [ ]:
# The old customer_id data is NOT resurrected!
print("Data after re-adding column (old data is NOT back):")
spark.sql("SELECT sale_id, product_name, customer_id FROM polaris.evolution.sales ORDER BY sale_id").show()

**What happened?**
- The original `customer_id` had field ID (e.g., 6)
- When we dropped it, that field ID was retired
- The new `customer_id` got a new field ID (e.g., 8)
- Old data files still contain field ID 6, but schema doesn't map it
- Result: old data is invisible, preventing accidental resurrection!

## Part 3: Renaming Columns

In [ ]:
# Rename a column
spark.sql("""
ALTER TABLE polaris.evolution.sales
RENAME COLUMN price TO unit_price
""")

print("Column renamed from 'price' to 'unit_price'")

In [ ]:
# Schema shows new name
print("Schema after rename:")
spark.sql("DESCRIBE polaris.evolution.sales").show()

In [ ]:
# Query using new name
print("Query using new name:")
spark.sql("""
SELECT product_name, unit_price, quantity
FROM polaris.evolution.sales
WHERE unit_price > 100
""").show()

**Key Point**: The field ID didn't change - only the name mapping changed. All existing data is still accessible.

## Part 4: Partition Evolution

### Create Unpartitioned Table

In [ ]:
# Create a table with event logs - initially unpartitioned
spark.sql("""
CREATE OR REPLACE TABLE polaris.evolution.events (
    event_id BIGINT,
    event_type STRING,
    user_id STRING,
    event_timestamp TIMESTAMP,
    event_data STRING
)
USING iceberg
""")

print("Events table created (unpartitioned)")

In [ ]:
# Insert some data
spark.sql("""
INSERT INTO polaris.evolution.events VALUES
    (1, 'login', 'user1', TIMESTAMP '2024-01-15 08:30:00', '{"device": "mobile"}'),
    (2, 'view', 'user1', TIMESTAMP '2024-01-15 08:31:00', '{"page": "home"}'),
    (3, 'click', 'user2', TIMESTAMP '2024-01-15 09:00:00', '{"button": "buy"}'),
    (4, 'login', 'user3', TIMESTAMP '2024-01-16 10:00:00', '{"device": "desktop"}'),
    (5, 'logout', 'user1', TIMESTAMP '2024-01-16 10:30:00', '{}'),
    (6, 'view', 'user2', TIMESTAMP '2024-01-17 14:00:00', '{"page": "products"}'),
    (7, 'purchase', 'user2', TIMESTAMP '2024-01-17 14:15:00', '{"amount": 99.99}')
""")

print("Event data inserted!")

### Check Initial Partition Spec

In [ ]:
# View partition spec
print("Initial partition spec:")
spark.sql("DESCRIBE EXTENDED polaris.evolution.events").filter(
    F.col("col_name").contains("Partition")
).show(truncate=False)

In [ ]:
# View files - should be unpartitioned
print("Files (unpartitioned):")
spark.sql("""
SELECT 
    file_path,
    record_count
FROM polaris.evolution.events.files
""").show(truncate=False)

### Add Partitioning to Existing Table

In [ ]:
# Add day partitioning (without rewriting existing data!)
spark.sql("""
ALTER TABLE polaris.evolution.events
ADD PARTITION FIELD days(event_timestamp)
""")

print("Partition field added!")

### Insert New Data - Now Partitioned

In [ ]:
# Insert more data - this will be partitioned
spark.sql("""
INSERT INTO polaris.evolution.events VALUES
    (8, 'login', 'user4', TIMESTAMP '2024-01-18 09:00:00', '{"device": "mobile"}'),
    (9, 'view', 'user4', TIMESTAMP '2024-01-18 09:05:00', '{"page": "home"}'),
    (10, 'login', 'user5', TIMESTAMP '2024-01-19 11:00:00', '{"device": "tablet"}'),
    (11, 'purchase', 'user5', TIMESTAMP '2024-01-19 11:30:00', '{"amount": 149.99}')
""")

print("New data inserted (will be partitioned)!")

### Compare Old vs New Files

In [ ]:
# View all files - mix of partitioned and unpartitioned
print("Files after partition evolution:")
spark.sql("""
SELECT 
    file_path,
    partition,
    record_count
FROM polaris.evolution.events.files
ORDER BY file_path
""").show(truncate=False)

**Observation**: Old files have empty partition values, new files are partitioned by day.

### Query Performance Comparison

In [ ]:
# Query for a specific day
print("Query for January 18, 2024:")
result = spark.sql("""
SELECT *
FROM polaris.evolution.events
WHERE event_timestamp >= TIMESTAMP '2024-01-18 00:00:00'
  AND event_timestamp < TIMESTAMP '2024-01-19 00:00:00'
""")
result.show()

In [ ]:
# Check scan metrics from Spark UI
# The query should skip old unpartitioned files based on min/max timestamps
# And efficiently target the Jan 18 partition for new files
print("Check Spark UI at http://localhost:4040 to see file pruning!")

### Change Partition Granularity

In [ ]:
# Change from day to hour partitioning
spark.sql("""
ALTER TABLE polaris.evolution.events
DROP PARTITION FIELD days(event_timestamp)
""")

spark.sql("""
ALTER TABLE polaris.evolution.events
ADD PARTITION FIELD hours(event_timestamp)
""")

print("Partition changed from days to hours!")

In [ ]:
# Insert more data with hour partitioning
spark.sql("""
INSERT INTO polaris.evolution.events VALUES
    (12, 'login', 'user6', TIMESTAMP '2024-01-20 08:00:00', '{"device": "mobile"}'),
    (13, 'view', 'user6', TIMESTAMP '2024-01-20 08:30:00', '{"page": "home"}'),
    (14, 'login', 'user7', TIMESTAMP '2024-01-20 09:00:00', '{"device": "desktop"}'),
    (15, 'purchase', 'user7', TIMESTAMP '2024-01-20 09:45:00', '{"amount": 199.99}')
""")

print("Data inserted with hour partitioning!")

In [ ]:
# View all files - now showing three different partition schemes
print("Files with mixed partition schemes:")
spark.sql("""
SELECT 
    file_path,
    partition,
    record_count
FROM polaris.evolution.events.files
ORDER BY file_path
""").show(truncate=False)

## Part 5: View Partition Spec History

In [ ]:
# View partition spec evolution
print("Partition spec history:")
spark.sql("""
SELECT 
    spec_id,
    partition,
    file_count,
    record_count
FROM polaris.evolution.events.partitions
""").show(truncate=False)

## Key Takeaways

### Schema Evolution
1. **Instant operations**: Add/drop/rename columns without rewriting data
2. **Field IDs**: Prevent data resurrection when re-adding columns
3. **Default values**: Handle schema mismatches transparently
4. **No downtime**: Schema changes don't block readers or writers

### Partition Evolution
1. **Add partitioning later**: Start unpartitioned, add partitions as needed
2. **Change granularity**: Switch from day to hour, or hour to day
3. **Mixed partitioning**: Old and new data coexist with different partition schemes
4. **No rewrite required**: Only new data uses the new partition spec

## Real-World Use Cases

- **Schema Evolution**: Add columns for new business requirements without downtime
- **Field IDs**: Safely delete PII columns knowing they won't accidentally come back
- **Partition Evolution**: Start with daily partitions, move to hourly as volume grows
- **Mixed Partitioning**: Historical data stays as-is while new data is optimized

## Challenge Exercise

1. Create a table with a basic schema
2. Add several columns with different default values
3. Drop and re-add a column to see field ID behavior
4. Evolve the partition spec twice
5. Query the table and observe file pruning in Spark UI

## Cleanup

In [ ]:
# Optional: Drop tables to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.sales")
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.events")
# print("Tables dropped!")